In [163]:
"""
Linear Regression Pipeline — WV Opioid Data
Target: LA_Opioid_Rate
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

In [164]:
df = pd.read_csv("../data/west_virginia_opioid_data_clean.csv")
df

,Year,FIPS_Code,State,County,Labor_Force_Participation_Rate,Unemployment_Rate,Median_Household_Income,Poverty_Percent_All_Ages,Poverty_Percent_Age_0_17,Pct_Never_Married,Pct_Divorced,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,Rural_Urban_Continuum_Code_2023,LA_Opioid_Rate,LA_Opioid_Prscrbng_Rate_1Y_Chg
0,2019,54001,West Virginia,Barbour,52.6,7.6,38459,20.8,30.8,48.3,8.0,10.9,47.3,15.6,9,4.89,-0.68
1,2020,54001,West Virginia,Barbour,53.0,8.4,38906,21.0,32.7,47.1,10.0,8.7,50.6,13.9,9,5.50,0.61
2,2021,54001,West Virginia,Barbour,50.3,10.0,42260,20.8,30.5,47.1,10.2,8.6,53.1,13.0,9,6.42,0.92
3,2022,54001,West Virginia,Barbour,49.9,10.1,44341,22.0,32.5,48.1,10.8,8.5,54.0,11.8,9,6.87,0.45
4,2023,54001,West Virginia,Barbour,50.0,10.3,48347,20.8,29.1,45.8,10.3,8.0,53.4,12.2,9,5.41,-1.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,2019,54109,West Virginia,Wyoming,39.9,13.4,42332,22.4,29.4,59.7,8.4,11.4,48.7,9.3,8,0.36,0.15
271,2020,54109,West Virginia,Wyoming,41.0,11.6,44095,21.4,29.8,62.6,7.7,13.5,46.6,11.8,8,0.36,0.15
272,2021,54109,West Virginia,Wyoming,36.9,8.0,44630,25.3,31.2,55.8,8.0,14.0,46.7,11.6,8,0.51,0.15
273,2022,54109,West Virginia,Wyoming,37.3,6.4,44510,24.4,31.3,51.7,9.4,15.8,46.6,11.4,8,0.94,0.43


In [165]:
# --- 2. Sort and engineer features ---
df_sorted = df.sort_values(['FIPS_Code', 'Year']).reset_index(drop=True)

# Lag feature: last year's rate
# For 2019 (no prior year), fill with the county's own 2019 rate to avoid dropping rows
df_sorted['rate_lag1'] = df_sorted.groupby('FIPS_Code')['LA_Opioid_Rate'].shift(1)
df_sorted['rate_lag1'] = df_sorted.groupby('FIPS_Code')['rate_lag1'].transform(
    lambda x: x.fillna(df_sorted.loc[x.index, 'LA_Opioid_Rate'])
)

# Target: next year's long-acting opioid rate
# Note: 2023 rows will have no target (no 2024 data) — these become prediction-only rows
df_sorted['target_next'] = df_sorted.groupby('FIPS_Code')['LA_Opioid_Rate'].shift(-1)

# Only drop rows missing the target (2023 rows are kept separately for final prediction)
df_model = df_sorted.dropna(subset=['target_next']).reset_index(drop=True)

print(f"Total rows before engineering: {len(df)}")
print(f"Rows available for modeling (2019–2022): {len(df_model)}")
print(f"Rows for 2023 prediction: {len(df_sorted[df_sorted['Year'] == 2023])}\n")


Total rows before engineering: 275
Rows available for modeling (2019–2022): 220
Rows for 2023 prediction: 55



In [166]:
df_sorted

,Year,FIPS_Code,State,County,Labor_Force_Participation_Rate,Unemployment_Rate,Median_Household_Income,Poverty_Percent_All_Ages,Poverty_Percent_Age_0_17,Pct_Never_Married,Pct_Divorced,Pct_Less_Than_HS,Pct_HS_Grad,Pct_Bachelors_Plus,Rural_Urban_Continuum_Code_2023,LA_Opioid_Rate,LA_Opioid_Prscrbng_Rate_1Y_Chg,rate_lag1,target_next
0,2019,54001,West Virginia,Barbour,52.6,7.6,38459,20.8,30.8,48.3,8.0,10.9,47.3,15.6,9,4.89,-0.68,4.89,5.50
1,2020,54001,West Virginia,Barbour,53.0,8.4,38906,21.0,32.7,47.1,10.0,8.7,50.6,13.9,9,5.50,0.61,4.89,6.42
2,2021,54001,West Virginia,Barbour,50.3,10.0,42260,20.8,30.5,47.1,10.2,8.6,53.1,13.0,9,6.42,0.92,5.50,6.87
3,2022,54001,West Virginia,Barbour,49.9,10.1,44341,22.0,32.5,48.1,10.8,8.5,54.0,11.8,9,6.87,0.45,6.42,5.41
4,2023,54001,West Virginia,Barbour,50.0,10.3,48347,20.8,29.1,45.8,10.3,8.0,53.4,12.2,9,5.41,-1.46,6.87,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,2019,54109,West Virginia,Wyoming,39.9,13.4,42332,22.4,29.4,59.7,8.4,11.4,48.7,9.3,8,0.36,0.15,0.36,0.36
271,2020,54109,West Virginia,Wyoming,41.0,11.6,44095,21.4,29.8,62.6,7.7,13.5,46.6,11.8,8,0.36,0.15,0.36,0.51
272,2021,54109,West Virginia,Wyoming,36.9,8.0,44630,25.3,31.2,55.8,8.0,14.0,46.7,11.6,8,0.51,0.15,0.36,0.94
273,2022,54109,West Virginia,Wyoming,37.3,6.4,44510,24.4,31.3,51.7,9.4,15.8,46.6,11.4,8,0.94,0.43,0.51,1.39


In [167]:
# --- 3. Define features explicitly ---
feature_cols = [
    'Labor_Force_Participation_Rate',
    'Unemployment_Rate',
    'Median_Household_Income',
    'Poverty_Percent_All_Ages',
    'Poverty_Percent_Age_0_17',
    'Pct_Never_Married',
    'Pct_Divorced',
    'Pct_Less_Than_HS',
    'Pct_HS_Grad',
    'Pct_Bachelors_Plus',
    'Rural_Urban_Continuum_Code_2023',
    'rate_lag1',
]

X = df_model[feature_cols]
y = df_model['target_next']


In [168]:
X.isna().sum()

Labor_Force_Participation_Rate     0
Unemployment_Rate                  0
Median_Household_Income            0
Poverty_Percent_All_Ages           0
Poverty_Percent_Age_0_17           0
Pct_Never_Married                  0
Pct_Divorced                       0
Pct_Less_Than_HS                   0
Pct_HS_Grad                        0
Pct_Bachelors_Plus                 0
Rural_Urban_Continuum_Code_2023    0
rate_lag1                          0
dtype: int64

In [169]:
# --- 5. Temporal train/test split: 2019–2021 train, 2022 test ---
# (2023 is held separately as real-world prediction set)
train_mask = df_model['Year'] <= 2021
test_mask  = df_model['Year'] == 2022

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train size: {len(X_train)} rows (2019–2021)")
print(f"Test size:  {len(X_test)} rows (2022)\n")

Train size: 165 rows (2019–2021)
Test size:  55 rows (2022)



In [170]:
# --- 6. Scale features (fit on train only) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [171]:
# --- 7. Train three models and compare ---
tscv = TimeSeriesSplit(n_splits=2)

# Model 1: Ridge
alphas = np.logspace(-3, 6, 100)
ridge = RidgeCV(alphas=alphas, cv=tscv)
ridge.fit(X_train_scaled, y_train)

# Model 2: Random Forest
rf = RandomForestRegressor(n_estimators=500, max_depth=6, min_samples_leaf=5, random_state=42)
rf.fit(X_train, y_train)

# Model 3: Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=500, max_depth=4, learning_rate=0.05,
                                min_samples_leaf=5, random_state=42)
gb.fit(X_train, y_train)


,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",500
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",4
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft

In [172]:
# --- 8. Evaluate all three ---
models = {
    'Ridge':             (ridge, X_test_scaled),
    'Random Forest':     (rf,    X_test),
    'Gradient Boosting': (gb,    X_test)
}

print("=" * 45)
print(f"{'Model':<22} {'R²':>7} {'RMSE':>8} {'MSE':>10}")
print("=" * 45)

results = {}
for name, (model, X_eval) in models.items():
    preds = model.predict(X_eval)
    r2   = r2_score(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    mse  = mean_squared_error(y_test, preds)
    results[name] = {'model': model, 'preds': preds, 'r2': r2, 'rmse': rmse, 'mse': mse}
    print(f"{name:<22} {r2:>7.4f} {rmse:>8.4f} {mse:>10.4f}")

print("=" * 45)

Model                       R²     RMSE        MSE
Ridge                   0.8125   1.7060     2.9105
Random Forest           0.7421   2.0007     4.0026
Gradient Boosting       0.7961   1.7790     3.1647


In [173]:
# --- 9. Pick best model by R² ---
best_name = max(results, key=lambda k: results[k]['r2'])
print(f"\nBest model: {best_name} (R²={results[best_name]['r2']:.4f})")


Best model: Ridge (R²=0.8125)


In [174]:
# --- 10. Feature importance from best model ---
print("\nFeature Importance:")
if best_name == 'Ridge':
    importance = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': np.abs(ridge.coef_)
    })
else:
    importance = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': results[best_name]['model'].feature_importances_
    })

importance = importance.sort_values('Importance', ascending=False)
print(importance.to_string(index=False))


Feature Importance:
                        Feature  Importance
                      rate_lag1    3.345249
 Labor_Force_Participation_Rate    0.611871
        Median_Household_Income    0.329369
              Unemployment_Rate    0.253985
               Pct_Less_Than_HS    0.166867
       Poverty_Percent_All_Ages    0.165168
             Pct_Bachelors_Plus    0.154533
              Pct_Never_Married    0.139985
                    Pct_HS_Grad    0.121974
       Poverty_Percent_Age_0_17    0.113078
                   Pct_Divorced    0.104991
Rural_Urban_Continuum_Code_2023    0.075013


In [175]:
# --- 11. Predict on 2023 and rank top 10 counties for intervention ---
# rate_lag1 for 2023 = each county's 2022 LA_Opioid_Rate
rate_2022 = df_sorted[df_sorted['Year'] == 2022].set_index('FIPS_Code')['LA_Opioid_Rate']

pred_rows = df_sorted[df_sorted['Year'] == 2023].copy().reset_index(drop=True)
pred_rows['rate_lag1'] = pred_rows['FIPS_Code'].map(rate_2022)

X_pred = pred_rows[feature_cols].fillna(pred_rows[feature_cols].median())

best_model  = results[best_name]['model']
X_pred_eval = scaler.transform(X_pred) if best_name == 'Ridge' else X_pred
pred_rows['predicted_rate'] = best_model.predict(X_pred_eval)

# Distress score: predicted rate + poverty + unemployment (all percentile ranked)
pred_rows['distress_score'] = (
    pred_rows['predicted_rate'].rank(pct=True) +
    pred_rows['Poverty_Percent_All_Ages'].rank(pct=True) +
    pred_rows['Unemployment_Rate'].rank(pct=True)
)

top10 = pred_rows[['FIPS_Code', 'State', 'County', 'predicted_rate',
                    'Poverty_Percent_All_Ages', 'Unemployment_Rate', 'distress_score']]\
    .sort_values('distress_score', ascending=False)\
    .head(10)\
    .reset_index(drop=True)

top10.index += 1
print(f"\nTop 10 Counties for Immediate Clinic Intervention (via {best_name}):")
print(top10.to_string())


Top 10 Counties for Immediate Clinic Intervention (via Ridge):
    FIPS_Code          State    County  predicted_rate  Poverty_Percent_All_Ages  Unemployment_Rate  distress_score
1       54013  West Virginia   Calhoun        5.759371                      33.4               10.9        2.781818
2       54001  West Virginia   Barbour        6.390021                      20.8               10.3        2.609091
3       54101  West Virginia   Webster        3.941364                      22.1                9.6        2.390909
4       54041  West Virginia     Lewis        5.762739                      19.4                9.4        2.363636
5       54059  West Virginia     Mingo        2.140903                      29.9               10.8        2.309091
6       54067  West Virginia  Nicholas        9.374843                      17.8                8.6        2.290909
7       54047  West Virginia  McDowell        1.537217                      30.9               14.9        2.218182
8       